# 임계값 최적화

In [ ]:
# ============================================================
# 0. 라이브러리
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    f1_score, recall_score, precision_score,
    roc_auc_score, average_precision_score, accuracy_score
)
from xgboost import XGBClassifier

# 불균형 처리 방식(Method)에 따라 필요한 라이브러리 (없으면 해당 방식만 사용 불가)
try:
    from imblearn.over_sampling import SMOTE, BorderlineSMOTE
    HAS_IMBLEARN = True
except ImportError:
    HAS_IMBLEARN = False

try:
    from ctgan import CTGAN
    HAS_CTGAN = True
except ImportError:
    HAS_CTGAN = False

warnings.filterwarnings("ignore")


# ============================================================
# 1. Top10_우수모델.csv 1번째 행 정보 로드
# ============================================================
TOP10_PATH = "15번. 우수모델 데이터/Top10 우수모델.csv"

top10 = pd.read_csv(TOP10_PATH, index_col=0)
row = top10.iloc[0]   # 1위 조합

FEATURE_SET  = row["FeatureSet"]
FEATURE_FILE = row["FeatureFile"]
METHOD       = row["Method"]
SMOTE_RATIO  = row["SMOTE_Ratio"]
MODEL_NAME   = row["Model"]

print("=" * 65)
print("Top10 1위 조합")
print(f"  FeatureSet  : {FEATURE_SET}")
print(f"  FeatureFile : {FEATURE_FILE}")
print(f"  Method      : {METHOD}")
print(f"  SMOTE_Ratio : {SMOTE_RATIO}")
print(f"  Model       : {MODEL_NAME}")
print("=" * 65)


# ============================================================
# 2. 설정값
# ============================================================
TRAIN_PATH   = r'10,11,12번\train데이터\M19_도매_소매업_train.parquet'
TEST_PATH    = r'10,11,12번\test데이터\M19_도매_소매업_test.parquet'
FEATURE_PATH = os.path.join(r'13번.피처셀렉션\M19_도매_소매업', FEATURE_FILE)

TARGET_COL   = "부실라벨_ICR3년"
RANDOM_STATE = 42
YEAR_COL     = "회계년도"
ID_COLS      = ["회사명", "사업자등록번호", "회계년도"]

FOLD_VAL_YEARS = [2016, 2017, 2018, 2019, 2020, 2021]
TRAIN_START    = 2012
RECALL_MIN     = 0.9   # ★ Recall >= 0.9 조건으로 threshold 탐색

SAVE_DIR = r'15번. 우수모델 데이터'
os.makedirs(SAVE_DIR, exist_ok=True)

PREFIX = FEATURE_SET   # 결과 파일명 접두사 (Top10의 다른 행도 돌릴 때 구분용)


# ============================================================
# 3. 데이터 로드
# ============================================================
train_full = pd.read_parquet(TRAIN_PATH)
test       = pd.read_parquet(TEST_PATH)

y_train_full = train_full[TARGET_COL]
y_test       = test[TARGET_COL]

feat_df      = pd.read_csv(FEATURE_PATH)
col_key      = "feature" if "feature" in feat_df.columns else feat_df.columns[0]
use_features = [f for f in feat_df[col_key].tolist() if f in train_full.columns]

_ref_pos_weight = (y_train_full == 0).sum() / (y_train_full == 1).sum()

print(f"\nTrain shape  : {train_full.shape}")
print(f"Test  shape  : {test.shape}")
print(f"피처 수       : {len(use_features)}개")
print(f"pos_weight(참고, 원본 분포) : {_ref_pos_weight:.4f}")
print("=" * 65)


# ============================================================
# 4. 모델 정의 (XGBoost, 주어진 하이퍼파라미터 + Method별 pos_weight)
# ============================================================
def make_model(pos_weight):
    return XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric="aucpr",
        random_state=RANDOM_STATE, verbosity=0,
        scale_pos_weight=pos_weight
    )


# ============================================================
# 5. Threshold 탐색 함수 (Recall >= RECALL_MIN을 만족하는 한계 threshold)
# ============================================================
def find_threshold_at_recall(y_true, y_prob, recall_min=RECALL_MIN):
    """
    Recall >= recall_min 을 만족하는 threshold들 중 '가장 큰' (가장 엄격한) threshold를 반환.
    -> threshold를 0.01부터 0.99까지 올리면 Recall은 단조 감소하므로,
       이 값이 Recall이 recall_min을 '막 넘기는(넘는 한계)' 지점이 됨.
    -> 만족하는 threshold가 하나도 없으면 None 반환.
    """
    thresholds = np.arange(0.01, 1.0, 0.01)
    valid = []
    for thr in thresholds:
        y_pred = (y_prob >= thr).astype(int)
        rec = recall_score(y_true, y_pred, zero_division=0)
        if rec >= recall_min:
            valid.append(round(thr, 2))
    if not valid:
        return None
    return max(valid)


def calc_metrics(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "F1"        : f1_score(y_true, y_pred, zero_division=0),
        "Recall"    : recall_score(y_true, y_pred, zero_division=0),
        "Precision" : precision_score(y_true, y_pred, zero_division=0),
        "ROC_AUC"   : roc_auc_score(y_true, y_prob),
        "PR_AUC"    : average_precision_score(y_true, y_prob),
        "Accuracy"  : accuracy_score(y_true, y_pred),
    }


# ============================================================
# 5-1. Method / SMOTE_Ratio 기반 불균형 처리 함수
#   - ClassWeight        : 리샘플링 없음, scale_pos_weight = (다수/소수)
#   - None / "-" / NaN   : 리샘플링 없음, scale_pos_weight = 1 (아무 처리도 안 함)
#   - SMOTE              : SMOTE 오버샘플링, scale_pos_weight = 1
#   - BorderlineSMOTE    : BorderlineSMOTE 오버샘플링, scale_pos_weight = 1
#   - CTGAN              : 소수 클래스를 CTGAN으로 학습 후 합성데이터 생성, scale_pos_weight = 1
#   - 그 외(알 수 없는 값): ClassWeight로 fallback (경고 출력)
# ============================================================
def _parse_smote_ratio(smote_ratio):
    """SMOTE_Ratio 값을 float으로 변환. '-' / NaN / 빈값 등은 None 반환."""
    try:
        if smote_ratio is None:
            return None
        s = str(smote_ratio).strip()
        if s in ("", "-", "nan", "None"):
            return None
        return float(s)
    except (TypeError, ValueError):
        return None


def _ctgan_resample(X, y, ratio):
    """소수 클래스(y==1) 표본을 CTGAN으로 학습해 합성 표본을 추가."""
    if not HAS_CTGAN:
        raise ImportError("CTGAN 방식을 사용하려면 'pip install ctgan'이 필요합니다.")

    X = X.reset_index(drop=True)
    y = y.reset_index(drop=True)

    minority = X[y == 1].copy()
    n_minority = len(minority)
    n_majority = (y == 0).sum()

    target_n = int(n_majority * ratio) if ratio else n_majority
    n_synth = max(0, target_n - n_minority)

    if n_synth == 0 or n_minority < 5:
        # 합성할 필요가 없거나 소수 표본이 너무 적으면 원본 그대로 반환
        return X, y

    ctgan = CTGAN(epochs=300, verbose=False)
    ctgan.fit(minority)
    synth = ctgan.sample(n_synth)
    synth = synth[X.columns]  # 컬럼 순서/구성 맞춤

    X_res = pd.concat([X, synth], ignore_index=True)
    y_res = pd.concat([y, pd.Series([1] * n_synth)], ignore_index=True)
    return X_res, y_res


def apply_resampling(X, y, method, smote_ratio):
    """
    Method/SMOTE_Ratio에 따라 (X_res, y_res, pos_weight)를 반환.
    - 리샘플링이 적용되면 클래스가 균형화되므로 pos_weight=1.0
    - 리샘플링이 없으면(ClassWeight) pos_weight = (다수/소수)
    """
    method_l = str(method).strip().lower()
    ratio = _parse_smote_ratio(smote_ratio)

    if method_l == "classweight":
        pos_weight = (y == 0).sum() / (y == 1).sum()
        return X, y, pos_weight

    if method_l in ("none", "", "-", "nan"):
        return X, y, 1.0

    if "borderline" in method_l:
        if not HAS_IMBLEARN:
            raise ImportError("BorderlineSMOTE 사용을 위해 'pip install imbalanced-learn'이 필요합니다.")
        sampler = BorderlineSMOTE(
            sampling_strategy=ratio if ratio is not None else "auto",
            random_state=RANDOM_STATE,
        )
        X_res, y_res = sampler.fit_resample(X, y)
        return X_res, y_res, 1.0

    if method_l == "smote":
        if not HAS_IMBLEARN:
            raise ImportError("SMOTE 사용을 위해 'pip install imbalanced-learn'이 필요합니다.")
        sampler = SMOTE(
            sampling_strategy=ratio if ratio is not None else "auto",
            random_state=RANDOM_STATE,
        )
        X_res, y_res = sampler.fit_resample(X, y)
        return X_res, y_res, 1.0

    if "ctgan" in method_l:
        X_res, y_res = _ctgan_resample(X, y, ratio)
        return X_res, y_res, 1.0

    # 알 수 없는 Method -> ClassWeight로 fallback
    print(f"  [경고] 알 수 없는 Method='{method}' -> ClassWeight(pos_weight)로 처리합니다.")
    pos_weight = (y == 0).sum() / (y == 1).sum()
    return X, y, pos_weight


# ============================================================
# 6. Expanding Window CV → fold별 확률 예측 수집 (threshold 탐색 없음)
# ============================================================
print(f"\n{'='*65}")
print("Expanding Window CV — fold별 확률 예측 수집")
print("=" * 65)

fold_results = []   # (val_year, y_true, y_prob) 저장 → 이후 Test 기준 THRESHOLD로 평가

for val_year in FOLD_VAL_YEARS:
    train_idx = train_full.index[
        (train_full[YEAR_COL] >= TRAIN_START) & (train_full[YEAR_COL] < val_year)
    ]
    val_idx = train_full.index[train_full[YEAR_COL] == val_year]

    if len(train_idx) == 0 or len(val_idx) == 0:
        print(f"  [경고] val_year={val_year} 데이터 없음 → skip")
        continue

    X_fold_train = train_full.loc[train_idx, use_features]
    y_fold_train = train_full.loc[train_idx, TARGET_COL]
    X_fold_val   = train_full.loc[val_idx,   use_features]
    y_fold_val   = train_full.loc[val_idx,   TARGET_COL]

    imputer      = SimpleImputer(strategy="median")
    X_fold_train = pd.DataFrame(imputer.fit_transform(X_fold_train), columns=use_features)
    X_fold_val   = pd.DataFrame(imputer.transform(X_fold_val),       columns=use_features)

    X_res, y_res, pos_weight = apply_resampling(X_fold_train, y_fold_train, METHOD, SMOTE_RATIO)

    model = make_model(pos_weight)
    model.fit(X_res, y_res)
    y_prob_val = model.predict_proba(X_fold_val)[:, 1]

    print(f"  Val {val_year} | ROC_AUC={roc_auc_score(y_fold_val, y_prob_val):.4f}  "
          f"PR_AUC={average_precision_score(y_fold_val, y_prob_val):.4f}")

    fold_results.append({"Val_Year": val_year, "y_true": y_fold_val.values, "y_prob": y_prob_val})


# ============================================================
# 7. 전체 train으로 최종모델 학습 → Test 예측 → Test 기준 THRESHOLD 산출
#    (Test 데이터에서 Recall >= RECALL_MIN을 만족하는 가장 엄격한 threshold)
# ============================================================
print(f"\n{'='*65}")
print(f"Test 데이터 기준 threshold 산출 (Recall >= {RECALL_MIN})")
print("=" * 65)

imputer_final = SimpleImputer(strategy="median")
X_train_all   = pd.DataFrame(
    imputer_final.fit_transform(train_full[use_features]), columns=use_features
)
X_test_imp    = pd.DataFrame(
    imputer_final.transform(test[use_features]), columns=use_features
)

X_train_res, y_train_res, _final_pos_weight = apply_resampling(
    X_train_all, y_train_full, METHOD, SMOTE_RATIO
)
print(f"  리샘플링 적용({METHOD}, SMOTE_Ratio={SMOTE_RATIO}): "
      f"{len(X_train_all)}행 -> {len(X_train_res)}행, pos_weight={_final_pos_weight:.4f}")

final_model = make_model(_final_pos_weight)
final_model.fit(X_train_res, y_train_res)
y_prob_test = final_model.predict_proba(X_test_imp)[:, 1]

found_threshold = find_threshold_at_recall(y_test, y_prob_test, RECALL_MIN)
if found_threshold is None:
    print(f"  [경고] Test 데이터에서 Recall >= {RECALL_MIN}을 만족하는 threshold가 없습니다. "
          f"threshold=0.01로 설정합니다.")
    THRESHOLD = 0.01
else:
    THRESHOLD = found_threshold

ref_m = calc_metrics(y_test, y_prob_test, THRESHOLD)
print(f"  >> Test 기준 threshold (Recall >= {RECALL_MIN}) = {THRESHOLD:.2f}")
print(f"     Recall={ref_m['Recall']:.4f}  Precision={ref_m['Precision']:.4f}  F1={ref_m['F1']:.4f}")


# ============================================================
# 8. CV fold별 성능 — Test 기준 THRESHOLD로 재계산 (참고용)
# ============================================================
cv_rows      = []
val_prob_all = []

for fr in fold_results:
    m = calc_metrics(fr["y_true"], fr["y_prob"], THRESHOLD)
    cv_rows.append({"Val_Year": fr["Val_Year"], **m})
    val_prob_all.append(pd.DataFrame({
        "Val_Year": fr["Val_Year"],
        "y_true"  : fr["y_true"],
        "y_prob"  : fr["y_prob"],
        "y_pred"  : (fr["y_prob"] >= THRESHOLD).astype(int),
    }))

cv_df   = pd.DataFrame(cv_rows).round(4)
cv_mean = cv_df[["F1", "Recall", "Precision", "ROC_AUC", "PR_AUC", "Accuracy"]].mean()
val_prob_df = pd.concat(val_prob_all, ignore_index=True)

print(f"\n{'='*65}")
print(f"CV fold별 성능 (threshold={THRESHOLD:.2f}, Test 기준 threshold 적용)")
print("=" * 65)
print(cv_df.to_string(index=False))
print(f"\n  CV 평균 | F1={cv_mean['F1']:.4f} Recall={cv_mean['Recall']:.4f} "
      f"Precision={cv_mean['Precision']:.4f} ROC_AUC={cv_mean['ROC_AUC']:.4f} "
      f"PR_AUC={cv_mean['PR_AUC']:.4f}")


# ============================================================
# 9. Test 평가 결과 출력
# ============================================================
print(f"\n{'='*65}")
print("Test 평가")
print("=" * 65)

y_pred_test = (y_prob_test >= THRESHOLD).astype(int)

test_metrics = calc_metrics(y_test, y_prob_test, THRESHOLD)
print(f"  Test (threshold={THRESHOLD:.2f}) | F1={test_metrics['F1']:.4f} "
      f"Recall={test_metrics['Recall']:.4f} Precision={test_metrics['Precision']:.4f} "
      f"ROC_AUC={test_metrics['ROC_AUC']:.4f} PR_AUC={test_metrics['PR_AUC']:.4f}")

print(f"\n  [Gap = Test - CV평균]")
for col in ["F1", "Recall", "Precision", "ROC_AUC", "PR_AUC", "Accuracy"]:
    gap = test_metrics[col] - cv_mean[col]
    print(f"    {col:<12} : {gap:+.4f}")


# ============================================================
# 10. 확률 분포 시각화 저장
# ============================================================
plt.rcParams.update({
    "font.family"       : "DejaVu Sans",
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
    "axes.grid"         : True,
    "grid.color"        : "#E5E5E5",
    "grid.linewidth"    : 0.7,
    "axes.facecolor"    : "#FAFAFA",
    "figure.facecolor"  : "white",
})

COLOR_NEG  = "#2F6EBA"
COLOR_POS  = "#D94F3D"
COLOR_THR  = "#F5A623"
ALPHA_HIST = 0.72
BINS       = 45

fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(
    2, 2, height_ratios=[3.2, 1],
    hspace=0.42, wspace=0.32,
    left=0.07, right=0.97, top=0.91, bottom=0.05
)
ax_cv   = fig.add_subplot(gs[0, 0])
ax_test = fig.add_subplot(gs[0, 1])
ax_tbl  = fig.add_subplot(gs[1, :])
ax_tbl.axis("off")

def plot_dist(ax, y_true, y_prob, title, n_total):
    arr0 = y_prob[np.array(y_true) == 0]
    arr1 = y_prob[np.array(y_true) == 1]

    counts0, edges0 = np.histogram(arr0, bins=BINS, range=(0, 1))
    counts1, edges1 = np.histogram(arr1, bins=BINS, range=(0, 1))

    ax.bar(edges0[:-1], counts0, width=np.diff(edges0),
           align="edge", color=COLOR_NEG, alpha=ALPHA_HIST, label="Normal (0)", zorder=3)
    ax.bar(edges1[:-1], counts1, width=np.diff(edges1),
           align="edge", color=COLOR_POS, alpha=ALPHA_HIST, label="Distress (1)", zorder=3)

    ax.axvline(THRESHOLD, color=COLOR_THR, linestyle="--",
               linewidth=1.8, zorder=5, label=f"Threshold = {THRESHOLD:.2f}")
    ax.axvspan(THRESHOLD, 1.0, alpha=0.06, color=COLOR_POS, zorder=2)

    n0, n1 = len(arr0), len(arr1)
    ir = n1 / n0 if n0 > 0 else float("nan")
    above_thr = (y_prob >= THRESHOLD).sum()
    stats_txt = (
        f"N={n_total:,}  |  Normal={n0:,}  Distress={n1:,}\n"
        f"Imbalance ratio = {ir:.3f}  |  Predicted Positive = {above_thr:,}"
    )
    ax.text(0.98, 0.97, stats_txt, transform=ax.transAxes, fontsize=8.2,
            va="top", ha="right",
            bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="#CCCCCC", alpha=0.85))

    ax.set_title(title, fontsize=12, fontweight="bold", pad=10)
    ax.set_xlabel("Predicted Probability", fontsize=9.5)
    ax.set_ylabel("Count", fontsize=9.5)
    ax.set_xlim(0, 1)
    ax.tick_params(labelsize=8.5)

    legend_elems = [
        Patch(facecolor=COLOR_NEG, alpha=ALPHA_HIST, label="Normal (0)"),
        Patch(facecolor=COLOR_POS, alpha=ALPHA_HIST, label="Distress (1)"),
        Line2D([0], [0], color=COLOR_THR, linestyle="--", linewidth=1.8,
               label=f"Threshold = {THRESHOLD:.2f}"),
    ]
    ax.legend(handles=legend_elems, fontsize=8.5, framealpha=0.9,
              loc="upper left", edgecolor="#CCCCCC")

plot_dist(ax_cv, val_prob_df["y_true"].values, val_prob_df["y_prob"].values,
          "Expanding Window CV — Predicted Probability Distribution",
          n_total=len(val_prob_df))
plot_dist(ax_test, y_test.values, y_prob_test,
          "Hold-out Test — Predicted Probability Distribution",
          n_total=len(y_test))

metrics_order = ["F1", "Recall", "Precision", "ROC_AUC", "PR_AUC", "Accuracy"]
col_labels    = ["Split"] + metrics_order

cv_row   = ["CV Mean"] + [f"{float(cv_mean[m]):.4f}" for m in metrics_order]
test_row = ["Test"]    + [f"{test_metrics[m]:.4f}"   for m in metrics_order]
gap_row  = ["Gap (Test − CV)"] + [
    f"{test_metrics[m] - float(cv_mean[m]):+.4f}" for m in metrics_order
]
table_data = [cv_row, test_row, gap_row]

tbl = ax_tbl.table(cellText=table_data, colLabels=col_labels, cellLoc="center", loc="center")
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1, 1.7)

for j in range(len(col_labels)):
    tbl[(0, j)].set_facecolor("#2F4F7F")
    tbl[(0, j)].set_text_props(color="white", fontweight="bold")

row_colors = ["#EEF3FA", "#FAFAFA", "#FFF4EE"]
for i, rc in enumerate(row_colors, start=1):
    for j in range(len(col_labels)):
        tbl[(i, j)].set_facecolor(rc)

for j, m in enumerate(metrics_order, start=1):
    gap_val = test_metrics[m] - float(cv_mean[m])
    color   = "#C0392B" if gap_val < -0.02 else ("#27AE60" if gap_val > 0.02 else "#555555")
    tbl[(3, j)].set_text_props(color=color, fontweight="bold")

ax_tbl.set_title("Performance Summary", fontsize=10, fontweight="bold", pad=6, loc="left")

fig.suptitle(
    f"XGBoost | {METHOD} | {FEATURE_SET} | M19 도매·소매업 | "
    f"Threshold(Recall≥{RECALL_MIN}) = {THRESHOLD:.2f}",
    fontsize=13.5, fontweight="bold", y=0.975
)

plt.savefig(os.path.join(SAVE_DIR, f"{PREFIX}_prob_distribution.png"), dpi=180, bbox_inches="tight")
plt.close()
print(f"\n  확률 분포 이미지 저장 완료 -> {PREFIX}_prob_distribution.png")


# ============================================================
# 11. Test 전체 행 예측 결과 저장
# ============================================================
test_id_cols = [c for c in ID_COLS if c in test.columns]
test_pred_df = test[test_id_cols].copy().reset_index(drop=True)
test_pred_df["y_true"]  = y_test.values
test_pred_df["y_prob"]  = y_prob_test.round(4)
test_pred_df["y_pred"]  = y_pred_test
test_pred_df["correct"] = (test_pred_df["y_true"] == test_pred_df["y_pred"]).astype(int)
test_pred_df["error_type"] = "TN"
test_pred_df.loc[(test_pred_df["y_true"]==1) & (test_pred_df["y_pred"]==1), "error_type"] = "TP"
test_pred_df.loc[(test_pred_df["y_true"]==1) & (test_pred_df["y_pred"]==0), "error_type"] = "FN"
test_pred_df.loc[(test_pred_df["y_true"]==0) & (test_pred_df["y_pred"]==1), "error_type"] = "FP"

print(f"\n  [Test 예측 분포]")
print(test_pred_df["error_type"].value_counts().to_string())

test_pred_df.to_csv(os.path.join(SAVE_DIR, f"{PREFIX}_test_predictions.csv"),
                     index=False, encoding="utf-8-sig")


# ============================================================
# 12. 결과 저장 — CV 결과 / 요약
# ============================================================
cv_df.to_csv(os.path.join(SAVE_DIR, f"{PREFIX}_CV_results.csv"), index=False, encoding="utf-8-sig")

summary = {
    "FeatureSet"       : FEATURE_SET,
    "FeatureFile"      : FEATURE_FILE,
    "N_Features"       : len(use_features),
    "Method"           : METHOD,
    "SMOTE_Ratio"      : SMOTE_RATIO,
    "Model"            : MODEL_NAME,
    "Recall_min"       : RECALL_MIN,
    "Threshold_basis"  : "Test",   # Test 데이터 기준으로 산출된 threshold
    "Threshold"        : THRESHOLD,
}
for col in ["F1", "Recall", "Precision", "ROC_AUC", "PR_AUC", "Accuracy"]:
    summary[f"CV_Val_{col}"] = round(float(cv_mean[col]), 4)
    summary[f"Test_{col}"]   = round(test_metrics[col], 4)
    summary[f"Gap_{col}"]    = round(test_metrics[col] - float(cv_mean[col]), 4)

pd.DataFrame([summary]).to_csv(os.path.join(SAVE_DIR, f"{PREFIX}_summary.csv"),
                                index=False, encoding="utf-8-sig")


# ============================================================
# 13. 2021~2024 PD 데이터 생성 및 저장
# ============================================================
PD_YEARS_2021_2024 = [2021, 2022, 2023, 2024]

all_data = pd.concat([train_full, test], ignore_index=True)
all_data_years = all_data[all_data[YEAR_COL].isin(PD_YEARS_2021_2024)].copy()

X_all_imp = pd.DataFrame(
    imputer_final.transform(all_data_years[use_features]),
    columns=use_features, index=all_data_years.index
)

all_data_years["y_prob"]  = final_model.predict_proba(X_all_imp)[:, 1].round(4)
all_data_years["y_pred"]  = (all_data_years["y_prob"] >= THRESHOLD).astype(int)
all_data_years["y_true"]  = all_data_years[TARGET_COL]
all_data_years["correct"] = (all_data_years["y_true"] == all_data_years["y_pred"]).astype(int)

all_data_years["error_type"] = "TN"
all_data_years.loc[(all_data_years["y_true"]==1)&(all_data_years["y_pred"]==1), "error_type"] = "TP"
all_data_years.loc[(all_data_years["y_true"]==1)&(all_data_years["y_pred"]==0), "error_type"] = "FN"
all_data_years.loc[(all_data_years["y_true"]==0)&(all_data_years["y_pred"]==1), "error_type"] = "FP"

id_cols_available = [c for c in ID_COLS if c in all_data_years.columns]
pd_df_2124 = all_data_years[id_cols_available + ["y_true","y_prob","y_pred","correct","error_type"]] \
    .sort_values([id_cols_available[1], YEAR_COL]).reset_index(drop=True)

pd_save_path_2124 = os.path.join(SAVE_DIR, f"2021_2024_PD_데이터.csv")
pd_df_2124.to_csv(pd_save_path_2124, index=False, encoding="utf-8-sig")

print(f"\n{'='*65}")
print(f"2021~2024 PD 데이터 저장 완료 -> {pd_save_path_2124}  ({len(pd_df_2124)}행)")
print(pd_df_2124[YEAR_COL].value_counts().sort_index().to_string())
print(pd_df_2124["error_type"].value_counts().to_string())


# ============================================================
# 14. 2014~2024 PD 데이터 생성 및 저장
# ============================================================
PD_YEARS_2014_2024 = list(range(2014, 2025))

all_data_years = all_data[all_data[YEAR_COL].isin(PD_YEARS_2014_2024)].copy()

X_all_imp = pd.DataFrame(
    imputer_final.transform(all_data_years[use_features]),
    columns=use_features, index=all_data_years.index
)

all_data_years["y_prob"]  = final_model.predict_proba(X_all_imp)[:, 1].round(4)
all_data_years["y_pred"]  = (all_data_years["y_prob"] >= THRESHOLD).astype(int)
all_data_years["y_true"]  = all_data_years[TARGET_COL]
all_data_years["correct"] = (all_data_years["y_true"] == all_data_years["y_pred"]).astype(int)

all_data_years["error_type"] = "TN"
all_data_years.loc[(all_data_years["y_true"]==1)&(all_data_years["y_pred"]==1), "error_type"] = "TP"
all_data_years.loc[(all_data_years["y_true"]==1)&(all_data_years["y_pred"]==0), "error_type"] = "FN"
all_data_years.loc[(all_data_years["y_true"]==0)&(all_data_years["y_pred"]==1), "error_type"] = "FP"

pd_df_1424 = all_data_years[id_cols_available + ["y_true","y_prob","y_pred","correct","error_type"]] \
    .sort_values([id_cols_available[1], YEAR_COL]).reset_index(drop=True)

pd_save_path_1424 = os.path.join(SAVE_DIR, f"2014_2024_PD_데이터.csv")
pd_df_1424.to_csv(pd_save_path_1424, index=False, encoding="utf-8-sig")

print(f"\n{'='*65}")
print(f"2014~2024 PD 데이터 저장 완료 -> {pd_save_path_1424}  ({len(pd_df_1424)}행)")
print(pd_df_1424[YEAR_COL].value_counts().sort_index().to_string())
print(pd_df_1424["error_type"].value_counts().to_string())
print("=" * 65)

print(f"\n저장 완료 (접두사 '{PREFIX}_')")
print(f"  → {PREFIX}_CV_results.csv")
print(f"  → {PREFIX}_summary.csv")
print(f"  → {PREFIX}_prob_distribution.png")
print(f"  → {PREFIX}_test_predictions.csv")
print(f"  → {PREFIX}_2021_2024_PD_데이터.csv")
print(f"  → {PREFIX}_2014_2024_PD_데이터.csv")
print(f"  ★ 최종 threshold (Recall>={RECALL_MIN}) = {THRESHOLD:.2f}")

Top10 1위 조합
  FeatureSet  : top65_dedup52
  FeatureFile : lasso_features_top65--52.csv
  Method      : ClassWeight
  SMOTE_Ratio : -
  Model       : XGBoost

Train shape  : (28111, 256)
Test  shape  : (11797, 256)
피처 수       : 52개
pos_weight(참고, 원본 분포) : 25.6455

Expanding Window CV — fold별 확률 예측 수집
  Val 2016 | ROC_AUC=0.9607  PR_AUC=0.4907
  Val 2017 | ROC_AUC=0.9590  PR_AUC=0.4028
  Val 2018 | ROC_AUC=0.9535  PR_AUC=0.3527
  Val 2019 | ROC_AUC=0.9584  PR_AUC=0.4695
  Val 2020 | ROC_AUC=0.9510  PR_AUC=0.3179
  Val 2021 | ROC_AUC=0.9701  PR_AUC=0.4881

Test 데이터 기준 threshold 산출 (Recall >= 0.9)
  리샘플링 적용(ClassWeight, SMOTE_Ratio=-): 28111행 -> 28111행, pos_weight=25.6455
  >> Test 기준 threshold (Recall >= 0.9) = 0.34
     Recall=0.9009  Precision=0.2649  F1=0.4094

CV fold별 성능 (threshold=0.34, Test 기준 threshold 적용)
 Val_Year     F1  Recall  Precision  ROC_AUC  PR_AUC  Accuracy
     2016 0.4218  0.7838     0.2886   0.9607  0.4907    0.9375
     2017 0.4094  0.7429     0.2826   0.9590  0.402